# Random walk, unit root, drift and trend: lab notebook

**Statistical Modelling and Forecasting, Statistics background for forecasting**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson

rng = np.random.default_rng(4)
n = 250

### 1. Three models side by side: how strongly does the series remember its past?

In [ ]:
# 1. Three models side by side: how strongly does the series remember its past?
e = rng.normal(size=n)
rw = np.cumsum(e)
rwd = np.cumsum(0.2 + e)
ts = 0.2 * np.arange(n) + e
tt = np.arange(n)
print("lag-1 autocorrelation of the deviations from a fitted straight line:")
for name, x in [("random walk", rw), ("random walk + drift", rwd), ("trend-stationary", ts)]:
    resid = x - np.polyval(np.polyfit(tt, x, 1), tt)
    print(f"  {name:20s} {np.corrcoef(resid[1:], resid[:-1])[0, 1]:6.3f}")

### 2. Unit-root tests with constant + trend (H0: unit root)

In [ ]:
# 2. Unit-root tests with constant + trend (H0: unit root)
# (adding a drift to a random walk does not change the ct statistic, because the trend term absorbs it)
for name, x in [("random walk", rw), ("random walk + drift", rwd), ("trend-stationary", ts)]:
    stat, p, *_ = adfuller(x, maxlag=1, regression="ct", autolag=None)
    print(f"ADF(ct) {name:20s} tau={stat:6.2f} p={p:.3f} -> {'reject unit root' if p < 0.05 else 'cannot reject unit root'}")

### 3. Forecast of a random walk with drift (numerical problem)

In [ ]:
# 3. Forecast of a random walk with drift (numerical problem)
yT, d, s, h = 120, 1.0, 3.0, 9
print("forecast:", yT + h * d, "| se:", s * np.sqrt(h), "| 95% half-width:", round(1.96 * s * np.sqrt(h), 2), "| Var(y_T), T=36:", 36 * s ** 2)

### 4. Empirical check of the variance growth

In [ ]:
# 4. Empirical check of the variance growth
paths = np.cumsum(rng.normal(size=(2000, 100)), axis=1)
print("variance at t=25, 50, 100:", np.round(paths[:, [24, 49, 99]].var(axis=0), 1), "(theory 25, 50, 100)")

### 5. Spurious regression: two independent random walks

In [ ]:
# 5. Spurious regression: two independent random walks
big = 0
r2s = []
for k in range(200):
    a = np.cumsum(rng.normal(size=n)); b = np.cumsum(rng.normal(size=n))
    fit = sm.OLS(b, sm.add_constant(a)).fit()
    big += abs(fit.tvalues[1]) > 2
    r2s.append(fit.rsquared)
print("share of runs with |t| > 2:", big / 200, "| mean R2:", round(float(np.mean(r2s)), 2))
fit = sm.OLS(b, sm.add_constant(a)).fit()
print("last run: R2", round(fit.rsquared, 2), "| Durbin-Watson", round(durbin_watson(fit.resid), 2))
# after differencing the relation disappears
fit2 = sm.OLS(np.diff(b), sm.add_constant(np.diff(a))).fit()
print("differenced regression: slope t-stat", round(fit2.tvalues[1], 2), "| R2", round(fit2.rsquared, 3))

## Expected output

Running all cells above should print:

```
lag-1 autocorrelation of the deviations from a fitted straight line:
  random walk           0.958
  random walk + drift   0.958
  trend-stationary     -0.072
ADF(ct) random walk          tau= -2.52 p=0.320 -> cannot reject unit root
ADF(ct) random walk + drift  tau= -2.52 p=0.320 -> cannot reject unit root
ADF(ct) trend-stationary     tau=-10.99 p=0.000 -> reject unit root
forecast: 129.0 | se: 9.0 | 95% half-width: 17.64 | Var(y_T), T=36: 324.0
variance at t=25, 50, 100: [ 25.3  51.1 102.4] (theory 25, 50, 100)
share of runs with |t| > 2: 0.845 | mean R2: 0.24
last run: R2 0.21 | Durbin-Watson 0.05
differenced regression: slope t-stat -0.87 | R2 0.003
```

## Practice exercises

Try these in the cells below.

1. Draw random walks and compare the fan of paths with the trend-stationary model. How does the spread over time differ?
2. Run the AR(1) option with φ = 0.98 and with φ = 1.0 (random walk). Can the ADF test tell them apart in a sample of 150?
3. Run the spurious regression 10 times. How often is the t-statistic larger than 2?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
